# YoctoGPT — a GPT in 100 lines of pure Python

No PyTorch, no NumPy — just `math` and `random`. We build a character-level Transformer language model from scratch, including a tiny autograd engine, attention, and a training loop.

This is an even simpler version of the [nanogpt lecture](https://www.youtube.com/watch?v=kCc8FmEb1nY) and [freeCodeCamp LLM course](https://www.youtube.com/watch?v=UU1WVnMk4E8). Everything lives in a single file with zero dependencies beyond the Python standard library.

---

## What You're About to See

By the end of this notebook, you'll watch a computer go from **pure gibberish** to **recognizable English** — learning to complete the phrase "Peter Piper picked a peck of pickled peppers" from scratch.

**Before training:**
```
Peter Piper 'sa'kIIeWkpp ep.sci.,cpk e osseIh
```

**After training:**
```
Peter Piper picked a peck of pickled peppers
```

The entire "brain" that learns this is just **~4,000 numbers** that start random and get slowly adjusted. That's it. No magic.

## How This Works (No Math Version)

Here's the entire idea in plain English:

1. **Start with random guesses** — The computer begins with ~4,000 random numbers. These numbers are its "brain."

2. **Make a prediction** — Given some letters like `"Peter P"`, it guesses what comes next. At first, it's terrible.

3. **Check the answer** — We compare its guess to the actual next letter (`"i"` in "Piper").

4. **Measure the mistake** — We calculate how wrong it was. This is called the **loss**.

5. **Adjust the numbers** — We figure out which of those 4,000 numbers to nudge up or down to make the guess slightly better. This is called **backpropagation**.

6. **Repeat 500 times** — Each time, the guesses get a tiny bit better.

7. **Watch it learn** — Gibberish → fragments → recognizable phrases.

That's it. Everything below is just the Python code to make this happen.

### The Big Picture

Here's the entire model at a glance — from input character to output prediction:

![Full GPT Model](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/06-full-gpt.png)

## Terminology Dictionary

If you see a scary ML term, look it up here:

| Term | Plain English |
|------|---------------|
| **Token** | A piece of text (for us, a single character like `'P'` or `' '`) |
| **Embedding** | A list of numbers that represents a token's "meaning" |
| **Parameter / Weight** | One of the adjustable numbers in the brain |
| **Forward pass** | Making a prediction |
| **Loss** | How wrong the prediction was (lower = better) |
| **Gradient** | Which direction to nudge each number to improve |
| **Backward pass** | Calculating all the gradients |
| **Learning rate** | How big of a nudge to make each step |
| **Attention** | The model looking at previous tokens to decide what's relevant |
| **Softmax** | Converting raw scores into probabilities (numbers 0-1 that sum to 1) |

Don't memorize these now — refer back as needed.

---
## 1 · Data & Tokenization

We use a tiny dataset — the "Peter Piper" tongue twister — so training finishes in minutes on a CPU. The tokenizer is character-level: every unique character gets an integer ID. We then split the encoded sequence 90/10 into training and validation sets.

**Why characters?** Real LLMs like ChatGPT use "subword tokens" (chunks like `"ing"` or `"the"`), but characters are simpler to understand. Same concept, just smaller pieces.

**Why split the data?** We hide 10% of the text during training. If the model can predict those unseen characters, it actually *learned the pattern* rather than just memorizing.

In [2]:
import math
import random

random.seed(67)

txt = "Peter Piper picked a peck of pickled peppers. A peck of pickled peppers Peter Piper picked. If Peter Piper picked a peck of pickled peppers, where's the peck of pickled peppers Peter Piper picked?"


In [11]:
len(txt)


196

In [9]:
# Build vocabulary: find all unique characters and sort them
characters = sorted(set(txt))   # e.g., ['\n', ' ', "'", 'A', 'P', 'a', ...]
vocab_size = len(characters)    # how many unique characters (24)

print(f"vocab size: {vocab_size}")
print(f"characters: {characters}")

vocab size: 23
characters: [' ', "'", ',', '.', '?', 'A', 'I', 'P', 'a', 'c', 'd', 'e', 'f', 'h', 'i', 'k', 'l', 'o', 'p', 'r', 's', 't', 'w']


In [10]:
# Encode: convert each character to its integer ID
# "Peter" → [8, 13, 23, 13, 21] (P=8, e=13, t=23, e=13, r=21)
token_ids = [characters.index(char) for char in txt]

print(f"encoded length: {len(token_ids)}")
print(f"first 50 tokens: {token_ids[:50]}")

encoded length: 196
first 50 tokens: [7, 11, 21, 11, 19, 0, 7, 14, 18, 11, 19, 0, 18, 14, 9, 15, 11, 10, 0, 8, 0, 18, 11, 9, 15, 0, 17, 12, 0, 18, 14, 9, 15, 16, 11, 10, 0, 18, 11, 18, 18, 11, 19, 20, 3, 0, 5, 0, 18, 11]


In [6]:
# Split into training (90%) and validation (10%)
split_point = int(len(token_ids) * 0.9)
train_data = token_ids[:split_point]  # first 90% — model learns from this
val_data = token_ids[split_point:]    # last 10% — we test on this (model never sees it during training)

print(f"train: {len(train_data)} tokens, val: {len(val_data)} tokens")

train: 177 tokens, val: 20 tokens


In [ ]:
# === TRY IT YOURSELF ===
# Uncomment these lines and run to experiment!

# What number is the letter 'P'?
# print(f"'P' is token number: {characters.index('P')}")

# Encode your own text:
# my_text = "Pepper"
# my_ids = [characters.index(c) for c in my_text]
# print(f"'{my_text}' encodes to: {my_ids}")

# Decode it back:
# decoded = ''.join([characters[i] for i in my_ids])
# print(f"Decoded back: '{decoded}'")

### Hyperparameters (settings we choose)

These are the "knobs" we set before training:

- `ctx = 8` — The model can only "see" the last 8 characters when predicting. (GPT-4 sees 128k+ tokens)
- `d = 16` — Each token becomes a 16-number vector. More numbers = more "meaning" capacity.
- `nl = 1` — Just one layer of attention. Real models have 32-96 layers.
- `lr = 0.01` — Learning rate. How big of a step to take each update.

In [7]:
# Hyperparameters — the "settings" we choose before training
context_length = 8     # how many characters the model can "see" at once
embed_dim = 16         # size of the vector representing each token
num_layers = 1         # how many attention + feedforward blocks
learning_rate = 0.01   # how big of a step to take each update

print(f"context={context_length}, embed_dim={embed_dim}, layers={num_layers}, lr={learning_rate}")

context=8, d_model=16, layers=1, lr=0.01


---
## 2 · Autograd Engine

**This is the hardest section conceptually. Go slow.**

### The Core Question: "If I tweak this number, how does the output change?"

Imagine you have a dial labeled `x` and a screen showing `y = x²`.
- When `x = 3`, the screen shows `y = 9`
- Nudge the dial to `x = 3.001`, the screen shows `y ≈ 9.006`
- The screen changed ~6× as fast as the dial

That ratio (≈6) is the **gradient**. It answers: *"Which direction should I turn the dial to make the screen's number go down?"*

**Why does this matter?** We have 4,000 dials (parameters). We need to know which way to nudge *all* of them to make the loss decrease. Computing this by hand would be insane — so we build a system that tracks it automatically.

### What class `V` does

The `V` class is a "smart number" that remembers how it was computed:
- `.data` = the actual number
- `.grad` = how much the final result changes when you tweak this number
- When you do `a + b` or `a * b`, the result remembers its "parents"
- Calling `.backward()` walks through every operation in reverse, filling in all gradients

This is exactly what PyTorch does under the hood. We're just building a tiny version. ([micrograd](https://github.com/karpathy/micrograd) is the inspiration)

### Visualizing the Computation Graph

Every operation creates a node in the graph. Gradients flow backward through it:

![Computation Graph](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/01-computation-graph.png)

In [12]:
# Let's prove the gradient idea manually BEFORE using class V:
# If y = x², then dy/dx = 2x. At x=3, that's 6.

x = 3.0
y = x ** 2  # y = 9

# Nudge x by a tiny amount
x_nudged = 3.001
y_nudged = x_nudged ** 2  # 9.006001

# How much did y change per unit change in x?
dy = y_nudged - y       # 0.006001
dx = 0.001
slope = dy / dx          # ≈ 6.0

print(f"Manual gradient: {slope:.4f}")
print(f"Calculus says:   {2 * x:.4f}")  # derivative of x² is 2x

Manual gradient: 6.0010
Calculus says:   6.0000


In [14]:
a = 3
b = 1
type(a)
# a + b

int

In [ ]:
class Value:
    """A 'smart number' that tracks how it was computed, enabling automatic gradient calculation."""
    
    def __init__(self, data, children=(), local_gradients=()):
        self.data = data                    # the actual number
        self.grad = 0.0                     # gradient: how much does the final output change if we tweak this?
        self.children = children            # what Values were used to compute this?
        self.local_grads = local_gradients  # how does this Value change w.r.t. each child?

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # d(a+b)/da = 1, d(a+b)/db = 1
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # d(a*b)/da = b, d(a*b)/db = a
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, n):
        # d(x^n)/dx = n * x^(n-1)
        return Value(self.data**n, (self,), (n * self.data ** (n - 1),))

    def __sub__(self, other):
        return self + (other * -1)

    def log(self):
        # d(log(x))/dx = 1/x
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def exp(self):
        # d(e^x)/dx = e^x
        result = math.exp(self.data)
        return Value(result, (self,), (result,))
        
    def relu(self):
        # d(relu(x))/dx = 1 if x > 0 else 0
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    def backward(self):
        """Compute gradients for all Values that contributed to this one."""
        # Build topological order (children before parents)
        topo_order, visited = [], set()
        def build_order(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    build_order(child)
                topo_order.append(node)
        build_order(self)
        
        # Backpropagate: walk backwards, accumulating gradients
        self.grad = 1.0  # d(output)/d(output) = 1
        for node in reversed(topo_order):
            for child, local_grad in zip(node.children, node.local_grads):
                child.grad += local_grad * node.grad  # chain rule!


In [9]:
# quick sanity check: d/dx (x^2) at x=3 should be 6
x = Value(3.0)
y = x ** 2
y.backward()
print(f"x={x.data}, y=x^2={y.data}, dy/dx={x.grad}")

x=3.0, y=x^2=9.0, dy/dx=6.0


**What just happened?** We computed `y = x²` using our `V` class, then called `y.backward()`. The gradient `x.grad = 6.0` tells us: *"If you increase x by 0.001, y will increase by about 0.006."* 

This is the same answer we got with the manual nudging above — but now it works automatically for *any* computation, no matter how complex. The entire GPT forward pass is just a bunch of adds and multiplies — `backward()` can handle all of it.

### Chain Rule in Action

When we have multiple operations, gradients multiply through the chain:

![Chain Rule](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/02-chain-rule.png)

---
## 3 · Model Parameters

**The "brain" is just a bunch of numbers organized into tables.**

We create several lookup tables and weight matrices, all filled with small random numbers:

| Name | Shape | Purpose |
|------|-------|---------|
| `wte` | 24 × 16 | **Token embeddings** — each of our 24 characters gets a 16-number "meaning vector" |
| `wpe` | 8 × 16 | **Position embeddings** — each position (0-7) gets a 16-number "location vector" |
| `lm` | 24 × 16 | **Language model head** — converts internal representation back to character probabilities |
| `q, k, v, o` | 16 × 16 | **Attention projections** — used for tokens to "look at" each other |
| `f1, f2` | 64 × 16, 16 × 64 | **Feed-forward network** — a small "thinking" layer |

**Total: ~4,000 numbers.** That's it. GPT-4 has ~1 trillion. Same idea, different scale.

### How Embeddings Work

Each character becomes a vector of learnable numbers:

![Token Embedding](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/03-token-embedding.png)

In [10]:
def make_matrix(rows, cols):
    """Create a rows × cols matrix of Value scalars, initialized with small random numbers."""
    # Small random numbers (mean=0, std=0.08) help training start smoothly
    return [[Value(random.gauss(0, 0.08)) for _ in range(cols)] for _ in range(rows)]

In [11]:
# Create all the weight matrices (the "brain")
weights = {
    "token_embed":    make_matrix(vocab_size, embed_dim),      # 24 × 16: each character gets a 16-number vector
    "position_embed": make_matrix(context_length, embed_dim),  # 8 × 16: each position gets a 16-number vector  
    "output_proj":    make_matrix(vocab_size, embed_dim),      # 24 × 16: convert back to character probabilities
}

# Add attention and feedforward weights for each layer
for layer_idx in range(num_layers):
    # Attention projections: Query, Key, Value, Output
    weights[f"layer{layer_idx}.query"]    = make_matrix(embed_dim, embed_dim)  # 16 × 16
    weights[f"layer{layer_idx}.key"]      = make_matrix(embed_dim, embed_dim)  # 16 × 16
    weights[f"layer{layer_idx}.value"]    = make_matrix(embed_dim, embed_dim)  # 16 × 16
    weights[f"layer{layer_idx}.attn_out"] = make_matrix(embed_dim, embed_dim)  # 16 × 16
    
    # Feedforward network: expand to 4x, then back down
    weights[f"layer{layer_idx}.ff_up"]   = make_matrix(embed_dim * 4, embed_dim)  # 64 × 16
    weights[f"layer{layer_idx}.ff_down"] = make_matrix(embed_dim, embed_dim * 4)  # 16 × 64

# Flatten all weights into a single list of parameters
all_parameters = [param for matrix in weights.values() for row in matrix for param in row]

print(f"total parameters: {len(all_parameters):,}")

total parameters: 3,968
weight matrices: ['wte', 'wpe', 'lm', 'l0.q', 'l0.k', 'l0.v', 'l0.o', 'l0.f1', 'l0.f2']


---
## 4 · Neural Net Primitives

Three small helper functions used everywhere in the model:

### `linear(x, w)` — Linear projection
Takes a vector `x` and a weight matrix `w`, returns `w × x`. This is how we transform information. If `x` is 16 numbers and `w` is 24×16, the output is 24 numbers.

### `normalize(x)` — Normalization  
Keeps numbers from getting too big or too small during training. Without this, values can "explode" to infinity or "vanish" to zero.

### `softmax(xs)` — Softmax
Converts a list of raw scores into probabilities (0 to 1, summing to 1). Bigger scores get bigger probabilities.

Example: `softmax([1, 2, 3])` → `[0.09, 0.24, 0.67]` (3 was biggest, so it gets the highest probability)

In [12]:
def linear(input_vector, weight_matrix):
    """
    Linear projection: multiply weight matrix by input vector.
    If input is 16 numbers and weight is 24×16, output is 24 numbers.
    Each output number is a weighted sum of all inputs.
    """
    output = []
    for row in weight_matrix:
        # Dot product: multiply corresponding elements and sum
        dot_product = sum((w * x for w, x in zip(row, input_vector)), Value(0))
        output.append(dot_product)
    return output

In [13]:
def normalize(vector):
    """
    RMS (Root Mean Square) normalization.
    Keeps numbers from exploding or vanishing during training.
    Divides each element by the RMS of the whole vector.
    """
    # Mean of squared values
    mean_squared = sum((x * x for x in vector), Value(0)) * (1 / len(vector))
    # Divide each element by sqrt(mean_squared), with small epsilon for stability
    scale = (mean_squared + 1e-5) ** -0.5
    return [x * scale for x in vector]

In [15]:
def softmax(scores):
    """
    Convert raw scores into probabilities (0 to 1, summing to 1).
    Bigger scores → bigger probabilities.
    
    Example: softmax([1, 2, 3]) → [0.09, 0.24, 0.67]
    """
    # Subtract max for numerical stability (prevents overflow in exp)
    max_score = max(v.data for v in scores)
    exp_scores = [(x - max_score).exp() for x in scores]
    
    # Normalize: divide each by sum so they add to 1
    total = sum(exp_scores, Value(0))
    if total.data == 0:
        return []
    return [x * (total ** -1) for x in exp_scores]

In [16]:
# quick test: softmax of [1, 2, 3] should sum to ~1
test_result = softmax([Value(1), Value(2), Value(3)])
print(f"softmax([1,2,3]) = {[round(v.data, 4) for v in test_result]}")
print(f"sum = {sum(v.data for v in test_result):.6f}")

softmax([1,2,3]) = [0.09, 0.2447, 0.6652]
sum = 1.000000


---
## 5 · GPT Forward Pass

This is the heart of the model. Given a token and its position, it predicts probabilities for the *next* token.

### The flow (simplified):

```
Token 'P' at position 0
    ↓
Look up embedding for 'P' (16 numbers)
Add position embedding for "slot 0" (16 numbers)  
    ↓
ATTENTION: "Look at all previous tokens, decide what's relevant"
    - Query: "What am I looking for?"
    - Keys:  "What does each previous token contain?"
    - Values: "What information to actually pass forward?"
    ↓
FEEDFORWARD: "Think about what I found"
    ↓
Output: 24 scores (one per possible next character)
    ↓
Softmax → Probabilities
    ↓
Prediction: "i" has highest probability (for "Pi" → "Piper")
```

### Why attention matters
Without attention, token 5 has no idea what tokens 0-4 said. With attention, it can "look back" and decide: *"Token 2 was a 'P' — that's relevant to my prediction."*

### Attention: "Which tokens matter?"

The model learns to focus on relevant previous tokens:

![Attention Mechanism](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/04-attention.png)

### The Transformer Block

One layer of the model — attention + feedforward with skip connections:

![GPT Block](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/05-gpt-block.png)

In [17]:
def gpt_forward(token_id, position, key_cache, value_cache):
    """
    Process one token and predict probabilities for the next token.
    
    Args:
        token_id: which character (0-23)
        position: where in the sequence (0-7)
        key_cache: stored keys from previous tokens (for attention)
        value_cache: stored values from previous tokens (for attention)
    
    Returns:
        logits: raw scores for each possible next character (24 numbers)
    """
    # Step 1: Look up embeddings and combine
    token_embedding = weights["token_embed"][token_id]      # 16 numbers representing "what token is this"
    position_embedding = weights["position_embed"][position] # 16 numbers representing "where is it"
    x = normalize([t + p for t, p in zip(token_embedding, position_embedding)])
    
    # Step 2: Process through each transformer layer
    for layer in range(num_layers):
        residual = x  # save for skip connection
        x = normalize(x)
        
        # Attention: let this token look at previous tokens
        query = linear(x, weights[f"layer{layer}.query"])   # "what am I looking for?"
        key   = linear(x, weights[f"layer{layer}.key"])     # "what do I contain?"
        value = linear(x, weights[f"layer{layer}.value"])   # "what info do I share?"
        
        # Add to cache so future tokens can attend to us
        key_cache[layer].append(key)
        value_cache[layer].append(value)
        
        # Compute attention scores: how relevant is each previous token?
        num_tokens = len(key_cache[layer])
        attention_scores = []
        for t in range(num_tokens):
            # Dot product between query and each key, scaled by sqrt(embed_dim)
            score = sum((query[j] * key_cache[layer][t][j] for j in range(embed_dim)), Value(0))
            score = score * (embed_dim ** -0.5)  # scale to prevent huge numbers
            attention_scores.append(score)
        
        attention_weights = softmax(attention_scores)  # convert to probabilities
        
        # Weighted sum of values based on attention
        attended = []
        for j in range(embed_dim):
            weighted_sum = sum((attention_weights[t] * value_cache[layer][t][j] 
                               for t in range(num_tokens)), Value(0))
            attended.append(weighted_sum)
        
        # Project and add residual connection
        attention_output = linear(attended, weights[f"layer{layer}.attn_out"])
        x = [a + r for a, r in zip(attention_output, residual)]
        
        # Feedforward network: expand, activate, contract
        ff_hidden = [h.relu() for h in linear(normalize(x), weights[f"layer{layer}.ff_up"])]
        ff_output = linear(ff_hidden, weights[f"layer{layer}.ff_down"])
        x = [a + b for a, b in zip(ff_output, x)]  # another residual connection
    
    # Step 3: Project to vocabulary size to get prediction scores
    logits = linear(x, weights["output_proj"])
    return logits

---
## 6 · Loss Function

**Loss = "How wrong was the prediction?"**

Here's the intuition:
- Model predicts: `{'P': 10%, 'i': 30%, 'e': 5%, ...}`  
- Actual next char: `'i'`
- We look up the probability it assigned to `'i'`: **30%**
- Loss = `-log(0.30)` ≈ **1.2**

**Why `-log`?**  
- If the model was confident and correct (90% on right answer): loss ≈ 0.1 (good!)  
- If the model was wrong (1% on right answer): loss ≈ 4.6 (bad!)  
- `-log` turns "probability of correct answer" into "how much to punish"

**What to expect:**
- Random guessing among 24 chars = 1/24 ≈ 4% → loss ≈ 3.2
- Well-trained model might get loss down to 0.5-1.0

In [18]:
def compute_loss(token_sequence):
    """
    Measure how well the model predicts the next token at each position.
    
    Lower loss = better predictions.
    - Loss ≈ 3.2 means random guessing (1/24 chance)
    - Loss ≈ 0.5 means pretty good predictions
    """
    num_predictions = min(context_length, len(token_sequence) - 1)
    
    # Initialize KV cache (stores keys/values for attention)
    key_cache = [[] for _ in range(num_layers)]
    value_cache = [[] for _ in range(num_layers)]
    
    total_loss = Value(0)
    
    for position in range(num_predictions):
        current_token = token_sequence[position]
        correct_next_token = token_sequence[position + 1]
        
        # Get model's predictions
        logits = gpt_forward(current_token, position, key_cache, value_cache)
        probabilities = softmax(logits)
        
        # How much probability did we assign to the correct answer?
        prob_of_correct = probabilities[correct_next_token]
        
        # Loss = -log(probability of correct answer)
        # High probability → low loss, low probability → high loss
        total_loss = total_loss + (prob_of_correct.log() * -1)
    
    # Return average loss per position
    return total_loss * (1 / num_predictions)

In [19]:
# Check initial loss — with random weights, expect loss ≈ ln(vocab_size)
# Why? Random model gives ~equal probability to all 24 chars: 1/24 ≈ 4%
# Loss = -log(1/24) = log(24) ≈ 3.18
expected_random_loss = math.log(vocab_size)
actual_initial_loss = compute_loss(val_data[:context_length + 1]).data

print(f"expected random loss: {expected_random_loss:.4f}")
print(f"actual initial loss:  {actual_initial_loss:.4f}")

expected random loss: 3.1781
actual initial loss:  3.3439


---
## 7 · Inference / Generation

**Generating text is just repeated prediction:**

1. Start with a prompt: `"Peter Piper "`
2. Feed it through the model → get probabilities for next character
3. **Sample** from those probabilities (not always pick the highest — adds variety)
4. Append the chosen character to the prompt
5. Repeat

**Before training:** The model's "brain" is random numbers, so it outputs gibberish.

**After training:** The same process produces recognizable text, because those numbers have been tuned to predict well.

In [20]:
def generate_text(prompt="Peter Piper ", num_chars=50):
    """
    Generate text by repeatedly predicting the next character.
    
    1. Start with prompt
    2. Feed through model → get probabilities for next char
    3. Sample from probabilities (not always pick highest — adds variety)
    4. Append new char, repeat
    """
    # Convert prompt to token IDs
    token_ids = [characters.index(c) for c in prompt]
    print(prompt, end="", flush=True)
    
    for _ in range(num_chars):
        # Fresh KV cache for each generation step
        key_cache = [[] for _ in range(num_layers)]
        value_cache = [[] for _ in range(num_layers)]
        
        # Feed the last `context_length` tokens through the model
        context = token_ids[-context_length:]
        for position, token in enumerate(context):
            logits = gpt_forward(token, position, key_cache, value_cache)
        
        # Convert logits to probabilities and sample
        probabilities = softmax(logits)
        prob_values = [p.data for p in probabilities]
        next_token = random.choices(range(vocab_size), weights=prob_values)[0]
        
        # Append and print
        token_ids.append(next_token)
        print(characters[next_token], end="", flush=True)
    
    print()  # newline at end

In [21]:
# generate before training — should be gibberish
print("Before training:")
generate_text()

Before training:
Peter Piper '
sa'kIIeWkpp
ep.sci.,cpk
e osseIh,?s?ikp.tdclsi
i


---
## 8 · Training Loop

**This is where learning happens.** 500 times, we:

1. **Pick a random chunk** of training text (8 characters)
2. **Forward pass:** Predict each next character, measure total loss
3. **Backward pass:** Call `backward()` to get gradients for all 4,000 parameters
4. **Update:** Nudge each parameter slightly in the direction that reduces loss

```
for each of 500 steps:
    loss = "how wrong was I on this chunk?"
    loss.backward()           # compute all gradients
    for each parameter p:
        p = p - learning_rate * gradient   # nudge toward better
```

**The optimizer trick (RMSprop):** We don't just use the raw gradient. We divide by a running average of recent gradient magnitudes. This helps parameters that get big gradients take smaller steps (prevents overshooting).

**Watch the loss drop:** ~3.3 (random) → ~0.5 (trained). The model goes from 4% accuracy to roughly 60%+ on predicting the next character.

### What to watch for

As training runs, look for:

1. **Loss going down:** ~3.3 → ~1.0 → ~0.5
2. **Text improving:** 
   - Step 0: Pure gibberish
   - Step 100: Some letters repeating (`"pep"`, `"per"`)
   - Step 300: Word fragments (`"pick"`, `"peck"`)
   - Step 500: Recognizable phrases

The model is literally learning English character patterns from scratch!

In [22]:
# ===== THE TRAINING LOOP =====
# This is where the magic happens: 500 iterations of "predict → measure error → adjust"

num_steps = 501

for step in range(num_steps):
    # Every 100 steps, check progress and generate sample text
    if step % 100 == 0:
        val_loss = compute_loss(val_data[:context_length + 1]).data
        print(f"\nstep {step}: val loss = {val_loss:.4f}")
        generate_text()
    
    # --- STEP 1: Pick a random chunk of training data ---
    start_idx = random.randint(0, len(train_data) - context_length - 1)
    chunk = train_data[start_idx : start_idx + context_length + 1]
    
    # --- STEP 2: Forward pass — compute loss ---
    current_loss = compute_loss(chunk)
    
    # --- STEP 3: Backward pass — compute gradients ---
    current_loss.backward()
    
    # --- STEP 4: Update parameters ---
    # Using RMSprop: divide gradient by running average of gradient magnitude
    # This prevents parameters with big gradients from updating too fast
    for param in all_parameters:
        # Update running average of squared gradient
        param.m = 0.99 * getattr(param, "m", 0.0) + 0.01 * (param.grad ** 2)
        
        # Update parameter: step in direction that reduces loss
        # Divide by sqrt(m) to normalize step size
        param.data = param.data - learning_rate * param.grad / (param.m ** 0.5 + 1e-8)
        
        # Reset gradient for next iteration
        param.grad = 0.0


step 0: val 3.3439
Peter Piper 'sdd kPdl?,?,eaes?sIikf'AsfktIac?f IkefilWI hc?fis

step 100: val 1.3527
Peter Piper pea per pepeck of ped per ppe's.
If pers.
s.
A pet

step 200: val 1.1328
Peter Piper pers of pe peppiteper er ppickleds,
If of Piperped

step 300: val 0.5346
Peter Piper Pick peck of pick of pick peck of pick of ped peck

step 400: val 0.6738
Peter Piper picked picked.
ed pickecked of picked peck of per 

step 500: val 0.4709
Peter Piper pickled peck of pirs.
A peper PPiper picked of pic


---
## What You Just Witnessed

You trained a Transformer language model from scratch. 

- **Step 0:** Random noise. `"Peter Piper 'sdd kPdl?,?"`
- **Step 500:** Recognizable patterns. `"Peter Piper picked a peck of..."`

The model learned that:
- `'P'` often follows `' '` (space)
- `"pick"` is a common substring
- `"peck"` and `"pepper"` are related patterns

All from adjusting 4,000 numbers based on "was I right about the next character?"

---

## What's the same in ChatGPT?

| This notebook | ChatGPT |
|---------------|---------|
| 4,000 parameters | ~1 trillion parameters |
| 8-character context | 128,000+ token context |
| Character-level tokens | Subword tokens (~50k vocabulary) |
| 1 attention head | 96+ heads per layer |
| Tongue twister dataset | Huge chunk of the internet |
| Same core idea: predict next token | Same core idea |

The architecture is the same. The scale is different.

---

## Next Steps

- **Go deeper:** [freeCodeCamp LLM course](https://www.youtube.com/watch?v=UU1WVnMk4E8) (7 hrs, uses PyTorch)
- **Original source:** [Karpathy's nanogpt lecture](https://www.youtube.com/watch?v=kCc8FmEb1nY) (2 hrs)
- **Make it fast:** [CUDA course](https://www.youtube.com/watch?v=86FAWCzIe_4) (12 hrs, GPU programming)
- **Understand backprop:** [Whiteboard explanation](https://youtu.be/0dbihoMRuyg?si=CvXDG6BC8khGcxoT)

In [ ]:
# === TRY IT YOURSELF ===
# The model is trained! Experiment with different prompts:

generate_text("A peck of ")
generate_text("If Peter ")
generate_text("pickled ")

# What happens with text the model never saw?
# generate_text("Hello ")